# 03 — Train/Test Distribution Shift

**Amaç:** Numeric ve kategorik dağılım farklarını, unseen category riskini belirlemek.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from kaggle_s6_e7.config import (FIGURES_DIR, TABLES_DIR, TARGET_COL, ID_COL,
    PLOT_SAMPLE_SIZE, RANDOM_STATE, ensure_report_dirs)
from kaggle_s6_e7.data import load_competition_data, infer_feature_columns, validate_schema
ensure_report_dirs()
train, test = load_competition_data()
validate_schema(train, test)
cat_cols, num_cols = infer_feature_columns(train)
plot_df = train.sample(min(PLOT_SAMPLE_SIZE, len(train)), random_state=RANDOM_STATE)
sns.set_theme(style="whitegrid")

## Numeric shift özeti

In [ ]:
from kaggle_s6_e7.eda import numeric_shift_summary, categorical_shift_summary
numeric_shift=numeric_shift_summary(train,test,num_cols)
display(numeric_shift)
numeric_shift.to_csv(TABLES_DIR / "03_numeric_shift.csv")
# Büyük örneklemde p-value yerine etki büyüklüğü olarak KS statistic önceliklidir.

## Numeric percentile karşılaştırması

In [ ]:
percentiles=[.001,.01,.05,.5,.95,.99,.999]
rows=[]
for col in num_cols:
    for source,df in [("train",train),("test",test)]:
        record={"feature":col,"source":source}
        record.update({f"q{q:g}":df[col].quantile(q) for q in percentiles})
        rows.append(record)
percentile_table=pd.DataFrame(rows)
display(percentile_table)
percentile_table.to_csv(TABLES_DIR / "03_numeric_percentiles.csv",index=False)

## Kategorik shift ve unseen category

In [ ]:
categorical_shift=categorical_shift_summary(train,test,cat_cols)
display(categorical_shift)
categorical_shift.to_csv(TABLES_DIR / "03_categorical_shift.csv")
for col in cat_cols:
    freq=pd.concat([train[col].value_counts(normalize=True,dropna=False).rename("train"),
                    test[col].value_counts(normalize=True,dropna=False).rename("test")],axis=1).fillna(0)
    print(f"\n{col}"); display(freq)

## Örneklemli görsel kontrol

In [ ]:
combined=pd.concat([train.assign(source="train"),test.assign(source="test")],ignore_index=True)
plot_combined=combined.groupby("source",group_keys=False).sample(n=min(50000,len(test)),random_state=RANDOM_STATE)
fig,axes=plt.subplots(len(num_cols),1,figsize=(9,3*len(num_cols)))
for col,ax in zip(num_cols,axes):
    sns.ecdfplot(data=plot_combined,x=col,hue="source",ax=ax)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "03_numeric_shift_ecdf.png",dpi=150); plt.show()

## Karar notu
KS p-value büyük örneklem nedeniyle aşırı hassastır; karar için KS statistic, quantile farkı ve görseller birlikte değerlendirilir.